In [1]:
import pandas as pd
from scipy.interpolate import interp1d
import numpy as np
import pyam

# Calculation of mineral demand for the all economy

## Clean USGS MCS data

In [60]:
df_usgs_mcs_raw = pd.read_csv(r'data/data_minerals/USGS/Mineral_commodity_summary_2025/MCS2025_World_Data.csv')

In [61]:
def restructure_mcs(df):
    # Step 1: Keep only rows where COUNTRY contains 'World'
    df_world = df[df['COUNTRY'].str.contains("World", case=False, na=False)].copy()
    df_world['COUNTRY'] = 'World total'

    # Step 2: Standardize column names for easier parsing
    df_world.rename(columns={
        'PROD_2023': '2023_PROD',
        'PROD_EST_ 2024': '2024_PROD',
        'PROD_NOTES': 'PROD_NOTES',
        'CAP_2023': '2023_CAP',
        'CAP_EST_ 2024': '2024_CAP',
        'CAP_NOTES': 'CAP_NOTES',
        'RESERVES_2024': '2024_RES',
        'RESERVE_NOTES': 'RES_NOTES'
    }, inplace=True)

    # Step 3: Melt the production, capacity, and reserves values
    value_df = pd.melt(
        df_world,
        id_vars=['COMMODITY', 'COUNTRY', 'TYPE', 'UNIT_MEAS'],
        value_vars=['2023_PROD', '2024_PROD', '2023_CAP', '2024_CAP', '2024_RES'],
        var_name='YEAR_DATATYPE',
        value_name='VALUE'
    )

    # Step 4: Extract YEAR and DATA_TYPE
    value_df[['YEAR', 'DATA_TYPE']] = value_df['YEAR_DATATYPE'].str.extract(r'(\d{4})_(PROD|CAP|RES)')
    value_df['DATA_TYPE'] = value_df['DATA_TYPE'].map({'PROD': 'Production', 'CAP': 'Capacity', 'RES': 'Reserves'})

    # Step 5: Melt the notes in long format and map their types
    note_df = pd.melt(
        df_world,
        id_vars=['COMMODITY', 'COUNTRY', 'TYPE', 'UNIT_MEAS'],
        value_vars=['PROD_NOTES', 'CAP_NOTES', 'RES_NOTES'],
        var_name='NOTE_COLUMN',
        value_name='NOTES'
    )
    note_df['DATA_TYPE'] = note_df['NOTE_COLUMN'].str.extract(r'(PROD|CAP|RES)')[0].map({'PROD': 'Production', 'CAP': 'Capacity', 'RES': 'Reserves'})
    note_df.drop(columns='NOTE_COLUMN', inplace=True)

    # Step 6: Merge values with corresponding notes
    merged_df = pd.merge(
        value_df,
        note_df,
        on=['COMMODITY', 'COUNTRY', 'TYPE', 'UNIT_MEAS', 'DATA_TYPE'],
        how='left'
    )

    # Step 7: Drop rows where VALUE is NaN and sort
    cleaned_df = merged_df.dropna(subset=['VALUE']).sort_values(by='COMMODITY').reset_index(drop=True)

    # Step 8: Return only relevant columns
    return cleaned_df[['COMMODITY', 'COUNTRY', 'TYPE', 'DATA_TYPE', 'UNIT_MEAS', 'YEAR', 'VALUE', 'NOTES']]

In [62]:
# Apply the function
df_usgs_mcs = restructure_mcs(df_usgs_mcs_raw)

In [59]:
df_usgs_mcs

,COMMODITY,COUNTRY,TYPE,DATA_TYPE,UNIT_MEAS,YEAR,VALUE,NOTES
0,Abrasives,World total,"Plant capacity, fused aluminum oxide",Capacity,metric tons,2023,1310000.0,NaN
1,Abrasives,World total,"Plant capacity, silicon carbide",Capacity,metric tons,2023,1010000.0,NaN
2,Abrasives,World total,"Plant capacity, fused aluminum oxide",Capacity,metric tons,2024,1310000.0,NaN
3,Abrasives,World total,"Plant capacity, silicon carbide",Capacity,metric tons,2024,1010000.0,NaN
4,Aluminum,World total,"Smelter production, aluminum",Production,thousand metric tons,2023,70000.0,NaN
...,...,...,...,...,...,...,...,...
223,Zinc,World total,"Mine production, zinc content",Production,thousand metric tons,2023,12100.0,NaN
224,Zinc,World total,"Mine production, zinc content",Reserves,thousand metric tons,2024,230000,NaN
225,Zirconium and Hafnium,World total,"Mine production, zirconium, gross weight",Production,thousand metric tons,2023,1440.0,NaN
226,Zirconium and Hafnium,World total,"Mine production, zirconium, gross weight",Production,thousand metric tons,2024,1500.0,NaN


In [65]:
df_usgs_mcs['COMMODITY'] = df_usgs_mcs['COMMODITY'].replace({
    "Zirconium and Hafnium": "Zirconium"})

In [66]:
df_usgs_mcs

,COMMODITY,COUNTRY,TYPE,DATA_TYPE,UNIT_MEAS,YEAR,VALUE,NOTES
0,Abrasives,World total,"Plant capacity, fused aluminum oxide",Capacity,metric tons,2023,1310000.0,NaN
1,Abrasives,World total,"Plant capacity, silicon carbide",Capacity,metric tons,2023,1010000.0,NaN
2,Abrasives,World total,"Plant capacity, fused aluminum oxide",Capacity,metric tons,2024,1310000.0,NaN
3,Abrasives,World total,"Plant capacity, silicon carbide",Capacity,metric tons,2024,1010000.0,NaN
4,Aluminum,World total,"Smelter production, aluminum",Production,thousand metric tons,2023,70000.0,NaN
...,...,...,...,...,...,...,...,...
223,Zinc,World total,"Mine production, zinc content",Production,thousand metric tons,2023,12100.0,NaN
224,Zinc,World total,"Mine production, zinc content",Reserves,thousand metric tons,2024,230000,NaN
225,Zirconium,World total,"Mine production, zirconium, gross weight",Production,thousand metric tons,2023,1440.0,NaN
226,Zirconium,World total,"Mine production, zirconium, gross weight",Production,thousand metric tons,2024,1500.0,NaN


In [74]:
df_usgs_mcs.to_csv(r'data/data_minerals/df_usgs_mcs.csv', index=False)

## Import clean datasets

In [37]:
# Get
#df = pd.read_excel(r'data/data_minerals/data_minerals.xlsx', sheet_name='CALCULATION')

# Get SSP data from IIASA database

In [ ]:
pyam.iiasa.platforms()

In [ ]:
conn = pyam.iiasa.Connection()
conn.valid_connections

In [19]:
conn_ssp = pyam.iiasa.Connection('ssp')

[INFO] 17:36:00 - pyam.iiasa: You are connected to the IXSE_SSP scenario explorer hosted by IIASA. If you use this data in any published format, please cite the data as provided in the explorer guidelines: https://data.ece.iiasa.ac.at/ssp/#/about
[INFO] 17:36:00 - pyam.iiasa: You are connected as an anonymous user


In [20]:
conn_ssp.models()

0       IIASA-WiC POP 2023
1     OECD ENV-Growth 2023
2    CDM Urbanization 2024
3           IIASA GDP 2023
Name: model, dtype: object

In [21]:
conn_ssp.scenarios()

0                      SSP1
1                      SSP2
2                      SSP3
3                      SSP4
4                      SSP5
5      Historical Reference
6     SSP1 - Review Phase 3
7     SSP2 - Review Phase 3
8     SSP3 - Review Phase 3
9     SSP4 - Review Phase 3
10    SSP5 - Review Phase 3
Name: scenario, dtype: object

In [22]:
conn_ssp.regions()

0                          World
1      Reforming Economies (R10)
2             Rest of Asia (R10)
3                    Other (R10)
4                          Aruba
                 ...            
272                 India+ (R10)
273          Latin America (R10)
274            Middle East (R10)
275          North America (R10)
276           Pacific OECD (R10)
Name: region, Length: 277, dtype: object

In [23]:
df_ssp = pyam.read_iiasa(
    "ssp",
    variable=["GDP|PPP", "GDP|PPP [per capita]", "Population", "Population|Urban|Share", "Population|Urban [Share]"],
    region="World",
    meta=True,
)

[INFO] 17:36:15 - ixmp4.data.backend.api: Connected to IXMP4 Platform 'ssp'
[WARNING] 17:36:15 - ixmp4.data.backend.api: IXMP4 Client and Server versions do not match. (Client: 0.9.8, Server: 0.9.6)
[INFO] 17:36:15 - ixmp4.data.backend.api: Platform notice: >
This platform has the "basic drivers", i.e., GDP and population projections, of the Shared Socioeconomic Pathways or "SSPs" (version 3.0.1, release March 2024). The projections are publicly available under a license that allows for the re-use by other research communities. Please visit https://data.ece.iiasa.ac.at/ssp for more information.


In [26]:
df_ssp.timeseries().reset_index().to_csv(r'data/data_ssp/iamdf_ssp.csv', index=False)

# Projection to 2050 using population or GDP

In [ ]:
## Get 

In [ ]:
## Project metal 

In [38]:
iamdf_ssp = pd.read_csv(r'data/data_ssp/iamdf_ssp.csv')

In [42]:
iamdf_ssp

,model,scenario,region,variable,unit,1950,1955,1960,1965,1970,...,2055,2060,2065,2070,2075,2080,2085,2090,2095,2100
0,IIASA GDP 2023,SSP2,World,GDP|PPP,billion USD_2017/yr,NaN,NaN,NaN,NaN,NaN,...,255138.335123,277925.973504,303738.457766,329904.061454,355181.883172,379416.234223,403091.576433,426844.242781,451003.054434,4.736919e+05
1,IIASA GDP 2023,SSP3,World,GDP|PPP,billion USD_2017/yr,NaN,NaN,NaN,NaN,NaN,...,225693.405454,237807.806366,251465.089082,263876.784353,274109.176024,282837.904227,291093.415351,299532.073994,308528.092060,3.176999e+05
2,IIASA GDP 2023,SSP4,World,GDP|PPP,billion USD_2017/yr,NaN,NaN,NaN,NaN,NaN,...,242718.489915,262626.219514,286025.130813,309511.569348,331753.809478,353275.893133,375617.507533,399219.452589,424297.929059,4.497758e+05
3,IIASA GDP 2023,SSP5,World,GDP|PPP,billion USD_2017/yr,NaN,NaN,NaN,NaN,NaN,...,303063.278481,339134.662984,381314.518550,422795.651999,460310.806649,492471.244082,519576.501571,542624.652729,564477.011532,5.835374e+05
4,IIASA-WiC POP 2023,Historical Reference,World,Population,million,2476.354507,2717.192825,2993.980903,3300.599684,3655.64095,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,IIASA-WiC POP 2023,SSP1,World,Population,million,NaN,NaN,NaN,NaN,NaN,...,9202.151029,9213.262946,9178.893434,9106.377915,8999.413476,8858.470328,8682.371633,8474.137111,8238.814653,7.979164e+03
6,IIASA-WiC POP 2023,SSP2,World,Population,million,NaN,NaN,NaN,NaN,NaN,...,9773.968144,9915.120022,10018.007006,10087.997781,10125.479114,10129.680830,10103.453315,10052.324966,9979.043839,9.884518e+03
7,IIASA-WiC POP 2023,SSP3,World,Population,million,NaN,NaN,NaN,NaN,NaN,...,10434.441507,10770.760005,11094.131841,11406.313579,11703.830762,11987.472742,12262.506162,12533.952723,12800.281382,1.305898e+04
8,IIASA-WiC POP 2023,SSP4,World,Population,million,NaN,NaN,NaN,NaN,NaN,...,10233.976397,10513.499547,10768.949236,11001.646474,11210.018164,11395.574578,11563.790683,11720.583524,11865.380260,1.199471e+04
9,IIASA-WiC POP 2023,SSP5,World,Population,million,NaN,NaN,NaN,NaN,NaN,...,9201.376985,9213.460664,9180.525398,9110.103583,9006.029578,8868.761464,8696.932656,8493.238260,8262.352940,8.006621e+03


In [46]:
def project_metal_data(df_yearbook, iamdf_ssp, model, scenario, variable, target_year=2050):
    """
    Projects mineral yearbook data using SSP-derived GDP or population growth rates.

    Parameters:
        df_yearbook (pd.DataFrame): Mineral Yearbook dataset.
        iamdf_ssp (pd.DataFrame): IAM SSP dataset with GDP/Population projections.
        model (str): The selected model (e.g., 'IIASA GDP 2023').
        scenario (str): The selected scenario (e.g., 'SSP2').
        variable (str): The selected variable ('GDP|PPP' or 'Population').
        target_year (int): The year up to which projections should be made.

    Returns:
        pd.DataFrame: Updated Yearbook dataset with projections.
    """
    projected_rows = []

    # Filter SSP dataset based on user selection
    ssp_filtered = iamdf_ssp[
        (iamdf_ssp["model"] == model) & 
        (iamdf_ssp["scenario"] == scenario) & 
        (iamdf_ssp["variable"] == variable) &
        (iamdf_ssp["region"] == "World")  # Ensure global values
    ]

    if ssp_filtered.empty:
        raise ValueError("No matching data found for the selected model, scenario, and variable.")

    # Extract available years and values for interpolation
    years_available = np.array([2020, 2025])
    values_available = ssp_filtered[[str(y) for y in years_available]].values.flatten()

    # Interpolate for missing years (linear interpolation)
    interp_func = interp1d(years_available, values_available, kind="linear", fill_value="extrapolate")
    interpolated_years = np.arange(2020, target_year + 1)
    interpolated_values = interp_func(interpolated_years)

    # Compute year-specific growth rates
    growth_rates = np.diff(interpolated_values) / interpolated_values[:-1]

    # Get latest available year in Yearbook data
    latest_year = df_yearbook["YEAR"].max()

    # Project mineral production using SSP growth rates
    for _, row in df_yearbook.iterrows():
        commodity, country, data_type, subtype, last_value, unit = (
            row["COMMODITY"], row["COUNTRY"], row["TYPE"], row["SUBTYPE"], row["VALUE"], row["UNIT"]
        )

        new_value = last_value
        for i, year in enumerate(range(latest_year + 1, target_year + 1)):
            growth_rate = growth_rates[min(i, len(growth_rates) - 1)]  # Ensure valid index
            new_value *= (1 + growth_rate)

            projected_rows.append({
                "SOURCE": "Mineral yearbook (Projected)",
                "COMMODITY": commodity,
                "COUNTRY": country,
                "YEAR": year,
                "TYPE": data_type,
                "SUBTYPE": subtype,
                "UNIT": unit,
                "VALUE": round(new_value, 2),
                "COMMENT": f"Projected using {variable} growth rate ({growth_rate:.2%})."
            })

    # Append projected data to original DataFrame
    df_projected = pd.concat([df_yearbook, pd.DataFrame(projected_rows)], ignore_index=True)
    df_projected = df_projected.sort_values(by=["COMMODITY", "YEAR"], ascending=[True, True])
    
    return df_projected


In [47]:
df_projected = project_mineral_yearbook(df_yearbook_updated, iamdf_ssp, model="IIASA-WiC POP 2023", scenario="SSP2", variable="Population")

In [48]:
df_projected

,SOURCE,COMMODITY,COUNTRY,YEAR,TYPE,SUBTYPE,UNIT,VALUE,COMMENT
0,Mineral yearbook,Aluminum,World,2021,Smelter production,Primary,kt,67500.00,Primary aluminum is defined as “The weight of ...
1,Mineral yearbook (Updated),Aluminum,World,2022,Smelter production,Primary,kt,68400.00,Extrapolated from 2021 using MCS growth rate f...
2,Mineral yearbook (Updated),Aluminum,World,2023,Smelter production,Primary,kt,70000.00,Extrapolated from 2022 using MCS growth rate f...
3,Mineral yearbook (Updated),Aluminum,World,2024,Smelter production,Primary,kt,72000.00,Extrapolated from 2023 using MCS growth rate f...
74,Mineral yearbook (Projected),Aluminum,World,2025,Smelter production,Primary,kt,68099.63,Projected using Population growth rate (0.89%).
...,...,...,...,...,...,...,...,...,...
1893,Mineral yearbook (Projected),Silicon,World,2050,Refinery production,Total|Ferrosilicon,t,14553050.53,Projected using Population growth rate (0.73%).
1919,Mineral yearbook (Projected),Silicon,World,2050,Refinery production,Total|Silicon metal,t,6527983.18,Projected using Population growth rate (0.73%).
1945,Mineral yearbook (Projected),Silicon,World,2050,Refinery production,Total,t,25928905.19,Projected using Population growth rate (0.73%).
1971,Mineral yearbook (Projected),Silicon,World,2050,Refinery production,Total|Ferrosilicon,t,17341251.79,Projected using Population growth rate (0.73%).


# IEA scenario data

In [4]:
df_iea = pd.read_csv(r'data/data_iea/WEO2024_AnnexA_Free_Dataset_World.csv')

In [6]:
# For CO2 calculations

# We keep only CO2 columns
filtered_categories = [
    "CO2 combustion",
]

filtered_product = [
    "Total"
]

filtered_flows = [
    "Total energy supply"
]

filtered_scenarios = [
    "Stated Policies Scenario",
    "Net Zero Emissions by 2050 Scenario"
]

filtered_year = [
    2022, 
    2023,
    2030,
    2035,
    2040,
    2050
]

df_iea_co2 = df_iea[df_iea["CATEGORY"].isin(filtered_categories)]
df_iea_co2 = df_iea_co2[df_iea_co2["PRODUCT"].isin(filtered_product)]
df_iea_co2 = df_iea_co2[df_iea_co2["FLOW"].isin(filtered_flows)]
df_iea_co2 = df_iea_co2[df_iea_co2["SCENARIO"].isin(filtered_scenarios)]
df_iea_co2 = df_iea_co2[df_iea_co2["YEAR"].isin(filtered_year)]
df_iea_co2

,PUBLICATION,SCENARIO,CATEGORY,PRODUCT,FLOW,UNIT,REGION,YEAR,VALUE
2398,World Energy Outlook 2024,Stated Policies Scenario,CO2 combustion,Total,Total energy supply,Mt CO2,World,2022,34290.31
2399,World Energy Outlook 2024,Stated Policies Scenario,CO2 combustion,Total,Total energy supply,Mt CO2,World,2023,34788.62
2400,World Energy Outlook 2024,Stated Policies Scenario,CO2 combustion,Total,Total energy supply,Mt CO2,World,2030,33231.60
2401,World Energy Outlook 2024,Stated Policies Scenario,CO2 combustion,Total,Total energy supply,Mt CO2,World,2035,30291.79
2402,World Energy Outlook 2024,Stated Policies Scenario,CO2 combustion,Total,Total energy supply,Mt CO2,World,2040,28163.34
2403,World Energy Outlook 2024,Stated Policies Scenario,CO2 combustion,Total,Total energy supply,Mt CO2,World,2050,25616.55
2409,World Energy Outlook 2024,Net Zero Emissions by 2050 Scenario,CO2 combustion,Total,Total energy supply,Mt CO2,World,2022,34290.31
2410,World Energy Outlook 2024,Net Zero Emissions by 2050 Scenario,CO2 combustion,Total,Total energy supply,Mt CO2,World,2023,34788.62
2411,World Energy Outlook 2024,Net Zero Emissions by 2050 Scenario,CO2 combustion,Total,Total energy supply,Mt CO2,World,2030,23082.09
2412,World Energy Outlook 2024,Net Zero Emissions by 2050 Scenario,CO2 combustion,Total,Total energy supply,Mt CO2,World,2035,12142.89


In [7]:
df_iea_co2.to_csv(r'data/data_iea/df_iea_co2.csv', index=False)